Use kernel: `Python (cellvit-infer)`.


In [1]:
import socket, torch
print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


host: hpctpa3pc0035
cuda: True
device_count: 1
gpu: NVIDIA A30


# Example for Inference

Be careful: The free instances just provide 2 CPU cores for your runtime. Thus,we need to set the hardware limits accordingly and overwrite them

First, download the test database

This also works for larger WSI

The results can be found in the test_results folder

## CellViT CLI sanity check (HPC)

Run this after `pixi run -e cellvit model-install` in your environment setup.


In [2]:
import subprocess
subprocess.run(["cellvit-check"], check=True)

2026-03-16 18:46:03,244 - Check-System - WARNING - To check the environment, the following python path is used:
2026-03-16 18:46:03,244 - Check-System - WARNING - /share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/bin/python

******************** Necessary Modules ********************

2026-03-16 18:46:03,244 - Check-System - INFO - Checking library torch
2026-03-16 18:46:03,245 - Check-System - INFO - Module installed and loaded
2026-03-16 18:46:03,245 - Check-System - INFO - Checking library torchaudio
2026-03-16 18:46:03,247 - Check-System - INFO - Module installed and loaded
2026-03-16 18:46:03,247 - Check-System - INFO - Checking library torchvision
2026-03-16 18:46:03,249 - Check-System - INFO - Module installed and loaded
2026-03-16 18:46:03,249 - Check-System - INFO - Checking library pandas
2026-03-16 18:46:03,251 - Check-System - INFO - Module installed and loaded
2026-03-16 18:46:03,251 - Check-System - INFO - Checking library numpy
2026-03-16 18:46:03,251 - Che

2026-03-16 18:46:15,654	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 
/share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/site-packages/ray/_private/worker.py:2067: FutureWarning: Pydantic v1 is deprecated and will no longer be supported in Ray 2.56. Please upgrade to Pydantic v2 by running `pip install pydantic>=2`. See https://github.com/ray-project/ray/issues/58876 for more details.
  warnings.warn(


[2026-03-16 18:46:06] [INFO] 
Main process sys.path:
[2026-03-16 18:46:06] [INFO]   - /share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/site-packages/ray/thirdparty_files
[2026-03-16 18:46:06] [INFO]   - /share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/site-packages/cellvit/utils
[2026-03-16 18:46:06] [INFO]   - /share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python310.zip
[2026-03-16 18:46:06] [INFO]   - /share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10
[2026-03-16 18:46:06] [INFO]   - /share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/lib-dynload
[2026-03-16 18:46:06] [INFO]   - /share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/site-packages
[2026-03-16 18:46:22] [INFO] 
Ray initialized successfully.
[2026-03-16 18:46:22] [INFO] 
Testing import in main process:
[2026-03-16 18:46:22] [INFO] CuPy is not available.
[2026-03-

CompletedProcess(args=['cellvit-check'], returncode=0)

## Example inference command (edit paths first)


In [3]:
!export CUDA_VISIBLE_DEVICES=0

In [5]:
## GPU version
from pathlib import Path

file='RCC_5'
OUTDIR = Path("/share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/")
# OUTDIR = Path("/share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/TCGA-BP-4977-01Z-00-DX1")
WSI_PATH = Path(f"/share/lab_soupir/IHC/ideas/HnE_segmentation/data/raw_HnE/SVS/{file}.svs")
# WSI_PATH = Path("/share/lab_teng/trainee/tusharsingh/cell-seg/data/wsi/TCGA-BP-4977-01Z-00-DX1.b40b5355-757d-42cc-91b2-333871afbb57.svs")
OUTDIR.mkdir(parents=True, exist_ok=True)
# OUTDIR.mkdir(parents=True, exist_ok=True)
MODELS = ["HIPT","SAM"]



for i in MODELS:
    cmd = [
        "cellvit-inference",
        "--model", i,      # "SAM" or "HIPT" in this wrapper
        "--gpu", "0",
        "--enforce_amp",
        "--batch_size", "4",
        "--outdir", str(OUTDIR / f"{i}"),
        "--geojson",
        "--cpu_count", "20",
        "--ray_worker", "4",
        "--ray_remote_cpus", "4",
        "process_wsi",
        "--wsi_path", str(WSI_PATH),
        "--wsi_mpp", "0.50",
        "--wsi_magnification", "20",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

cellvit-inference --model HIPT --gpu 0 --enforce_amp --batch_size 4 --outdir /share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/HIPT --geojson --cpu_count 20 --ray_worker 4 --ray_remote_cpus 4 process_wsi --wsi_path /share/lab_soupir/IHC/ideas/HnE_segmentation/data/raw_HnE/SVS/RCC_5.svs --wsi_mpp 0.50 --wsi_magnification 20


INFO     | SystemConfiguration  | ========================================
INFO     | SystemConfiguration  |          SYSTEM CONFIGURATION           
INFO     | SystemConfiguration  | ========================================
INFO     | SystemConfiguration  | System Configuration:
INFO     | SystemConfiguration  | CPU count:          20
INFO     | SystemConfiguration  | Memory:             128.00 GB
INFO     | SystemConfiguration  | GPU count:          1
INFO     | SystemConfiguration  | Used GPU-ID:        0
INFO     | SystemConfiguration  | GPU memory:         25.34 GB
INFO     | SystemConfiguration  | Ray available:      True
INFO     | SystemConfiguration  | Ray worker count:   4
INFO     | SystemConfiguration  | Ray remote cpus:    4
INFO     | SystemConfiguration  | Cupy available:     False
INFO     | SystemConfiguration  | Cucim available:    False
INFO     | SystemConfiguration  | Numba available:    True
INFO     | SystemConfiguration  | =======================================




2026-03-16 19:12:04,096 [INFO] - Loading model: HIPT
2026-03-16 19:12:04,097 [INFO] - The file CellViT-256-x40-AMP.pth already exists in /home/80033718/.cache/cellvit.
2026-03-16 19:12:04,491 [INFO] - <All keys matched successfully>
2026-03-16 19:12:04,683 [INFO] - Based on the hardware we limit the batch size to a maximum of:
2026-03-16 19:12:04,683 [INFO] - 8
2026-03-16 19:12:04,683 [INFO] - Using default PanNuke classes
2026-03-16 19:12:04,683 [INFO] - Loading inference transformations


2026-03-16 19:12:10,959	INFO worker.py:2013 -- Started a local Ray instance.
/share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/share/lab_teng/trainee/tusharsingh/cell-seg/.pixi/envs/cellvit/lib/python3.10/site-packages/ray/_private/worker.py:2067: FutureWarning: Pydantic v1 is deprecated and will no longer be supported in Ray 2.56. Please upgrade to Pydantic v2 by running `pip install pydantic>=2`. See https://github.com/ray-project/ray/issues/58876 for more details.
  warnings.warn(


2026-03-16 19:12:12,877 [INFO] - Processing single WSI file
2026-03-16 19:12:12,877 [INFO] - Processing WSI: RCC_5.svs
2026-03-16 19:12:12,878 [INFO] - Preparing WSI - Loading tissue region and prepare patches
2026-03-16 19:12:13,174 [WARNING] - Requested mpp resolution (0.25) is not a power of the base resolution 0.5. We perform rescaling, but this may not be accurate and is very slow!
2026-03-16 19:12:13,174 [WARNING] - We need to rescale to 0.25000, handle with care! Check the final results!
2026-03-16 19:12:13,727 [WARNING] - Requested mpp resolution (0.25) is not a power of the base resolution 0.5. We perform rescaling, but this may not be accurate and is very slow!
2026-03-16 19:12:20,574 [WARNING] - The given setting results in the following specifications for background detection: downsampling-patch-size=30.0, downsampling-overlap: 1.0 This could cause a slight shift between the background detection grid and the actual patches extracted. Grid is interpolated and rearranged, but

...

2026-03-16 19:17:14,228 [INFO] - Processing single WSI file
2026-03-16 19:17:14,228 [INFO] - Processing WSI: RCC_5.svs
2026-03-16 19:17:14,229 [INFO] - Preparing WSI - Loading tissue region and prepare patches
2026-03-16 19:17:14,517 [WARNING] - Requested mpp resolution (0.25) is not a power of the base resolution 0.5. We perform rescaling, but this may not be accurate and is very slow!
2026-03-16 19:17:14,517 [WARNING] - We need to rescale to 0.25000, handle with care! Check the final results!
2026-03-16 19:17:15,078 [WARNING] - Requested mpp resolution (0.25) is not a power of the base resolution 0.5. We perform rescaling, but this may not be accurate and is very slow!
2026-03-16 19:17:21,725 [WARNING] - The given setting results in the following specifications for background detection: downsampling-patch-size=30.0, downsampling-overlap: 1.0 This could cause a slight shift between the background detection grid and the actual patches extracted. Grid is interpolated and rearranged, but

100%|█| 204/204 [06:20<00:00,  2.09s/it, mem=10.73%, status=Running CellViT...]

2026-03-16 19:23:42,580 [INFO] - Waiting for final batches to be processed...


100%|█| 204/204 [06:21<00:00,  1.87s/it, mem=10.73%, status=Running CellViT...]


2026-03-16 19:23:43,504 [INFO] - Unpack Batches
2026-03-16 19:23:43,784 [INFO] - Initializing Cell-Postprocessor
2026-03-16 19:23:44,101 [INFO] - Finding edge-cells for merging
2026-03-16 19:23:44,357 [INFO] - Removal of cells detected multiple times
2026-03-16 19:23:47,894 [INFO] - Iteration 0: Found overlap of # cells: 7694
2026-03-16 19:23:49,605 [INFO] - Iteration 1: Found overlap of # cells: 76
2026-03-16 19:23:51,350 [INFO] - Iteration 2: Found overlap of # cells: 0
2026-03-16 19:23:51,350 [INFO] - Found all overlapping cells
2026-03-16 19:23:51,472 [INFO] - Detected cells after cleaning: 97768
2026-03-16 19:23:55,643 [INFO] - Converting segmentation to geojson
2026-03-16 19:23:56,802 [INFO] - Converting detection to geojson
2026-03-16 19:23:57,098 [INFO] - Finished with cell detection for WSI RCC_5
2026-03-16 19:23:57,099 [INFO] - Stats:
2026-03-16 19:23:57,099 [INFO] - {'Connective': 55841, 'Neoplastic': 26317, 'Inflammatory': 15533, 'Epithelial': 77}
2026-03-16 19:23:57,710 [I